# TerraMind Multitemporal Rice Fine-tuning — Blueprint Final

Notebook นี้เป็น **Fine-tuning Blueprint สำหรับ Google Colab** ที่ออกแบบจากบทเรียนทั้งหมดของโครงการตรวจจับนาข้าวด้วย TerraMind

เป้าหมายมีสองระดับพร้อมกัน:

1. **Showcase แรก**  
   ใช้ dataset และ checkpoint ที่เตรียมไว้ใน Google Drive แล้ว resume ต่ออีก 2–3 epochs เพื่อพิสูจน์ว่า workflow ตั้งแต่ input → train → validation → checkpoint → evaluation ทำงานได้จริง

2. **Blueprint สำหรับงานในอนาคต**  
   บริษัทสามารถเปลี่ยน dataset, เปลี่ยน checkpoint, เริ่มจาก pretrained TerraMind บน Hugging Face หรือปรับค่าการ train ได้จาก configuration cell เพียงจุดเดียว

---

## แนวคิดสำคัญ

- ใช้ `gmount` เท่านั้น ไม่ใช้ `gdown`
- ทุก path อ้างอิงจาก `PROJECT_ROOT`
- รองรับทั้ง `resume`, `pretrained` และ `auto`
- Dataset อ่านแบบ lazy จาก HDF5 ไม่โหลดทั้งชุดเข้า RAM
- Copy dataset ไป local SSD ของ Colab เมื่อพื้นที่เพียงพอ
- GPU required เพื่อป้องกันการ train บน CPU โดยไม่ตั้งใจ
- บันทึก `best.ckpt`, `last.ckpt`, configuration, normalization, logs และ metrics กลับ Google Drive
- ตรวจ compatibility ของ dataset กับ checkpoint ก่อนเริ่ม
- มี smoke test ก่อนใช้เวลาหลายนาทีหรือหลายชั่วโมง
- หลัง smoke test โมเดลถูกคืนเป็น `train()` อย่างชัดเจน
- ไม่ pin NumPy เป็น `1.26.4` เพื่อหลีกเลี่ยง dependency conflict
- ใช้ `importlib.metadata.version()` แทน `terratorch.__version__`

---

## Dataset contract

Dataset ต้องเป็น HDF5 shards และมีโครงสร้างต่อ sample:

```text
images             [T, C, H, W]
valid_masks        [T, H, W]
timestep_quality   [T]
labels             [H, W]
```

ข้อมูลเสริมต่อไปนี้รองรับแต่ไม่บังคับ:

```text
imputed_masks      [T, H, W]
timestamps         [T]
band_names         [C]
```

Label:

```text
0   = non-rice
1   = rice
255 = ignore / uncertain
```

Model input หลังผ่าน Dataset class:

```text
[B, C, T, H, W]
```

> Notebook นี้ทำ semantic segmentation สองคลาส ไม่ได้สร้าง training labels จากภาพดิบ หากบริษัทเปลี่ยน dataset ต้องเตรียม HDF5 ตาม contract ข้างต้นก่อน

# 0 — ก่อนเริ่ม

ใน Google Colab เลือก:

```text
Runtime → Change runtime type → GPU
```

แนะนำ:

- T4: ใช้งาน showcase ได้
- L4: เร็วกว่าและรองรับ BF16
- A100: เหมาะกับ training ขนาดใหญ่

รัน Cell 1 เพื่อติดตั้ง package จากนั้นดำเนินการตามลำดับจากบนลงล่าง  
หาก Colab แจ้งว่า package ถูกโหลดก่อนติดตั้ง ให้ Restart session แล้วเริ่มจาก Cell 2

In [ ]:
# ============================================================
# 01 — INSTALL DEPENDENCIES
# ============================================================

%pip -q install --upgrade --upgrade-strategy only-if-needed --no-cache-dir \
    "terratorch==1.2.10" \
    "lightning>=2.5,<2.7" \
    "huggingface_hub>=0.27" \
    "h5py>=3.10" \
    "pandas>=2.1" \
    "matplotlib>=3.8" \
    "tqdm>=4.66" \
    "scikit-learn>=1.4" \
    "tensorboard>=2.16"

# 1 — Import, reproducibility และ environment

Cell นี้:

- import library ที่ใช้ทั้งหมด
- ตั้ง random seed
- แสดง version ของ environment
- ตรวจว่า PyTorch เป็น CUDA build และ GPU พร้อมใช้งาน
- เปิด TF32 บน GPU ที่รองรับเพื่อเพิ่มประสิทธิภาพ

Notebook จะหยุดทันทีเมื่อไม่พบ GPU แทนที่จะเผลอ train บน CPU

In [ ]:
# ============================================================
# 02 — IMPORT + ENVIRONMENT
# ============================================================

from __future__ import annotations

from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import copy
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
import terratorch

from huggingface_hub import hf_hub_download
from lightning.pytorch.callbacks import (
    Callback,
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


SEED = 42
pl.seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision("high")


def package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "unknown"


print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Lightning:", pl.__version__)
print("TerraTorch:", package_version("terratorch"))
print("HDF5:", h5py.__version__)


if not torch.cuda.is_available():
    raise RuntimeError(
        "ไม่พบ CUDA GPU\n"
        "เลือก Runtime > Change runtime type > GPU แล้วเริ่มใหม่"
    )


DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print("GPU:", GPU_NAME)
print("GPU memory:", f"{GPU_MEMORY_GB:.2f} GB")
print("CUDA capability:", GPU_CAPABILITY)

# 2 — Mount Google Drive

หัวหน้าหรือผู้ใช้คนอื่นต้อง:

1. ได้รับสิทธิ์เข้าถึงโฟลเดอร์ project
2. Add shortcut ของโฟลเดอร์เข้า **My Drive**
3. Mount Google Drive ใน Colab

Notebook จะมอง shortcut เหมือนโฟลเดอร์ใต้:

```text
/content/drive/MyDrive/
```

In [ ]:
# ============================================================
# 03 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True,
)

print("Google Drive mounted")

# 3 — User configuration: จุดเดียวที่ควรแก้

สำหรับ showcase แรก ค่า default ถูกตั้งให้:

- ใช้ project folder ที่แชร์ผ่าน Google Drive
- resume จาก `last.ckpt`
- fine-tune ต่ออีก `3` epochs
- ใช้ dataset เดิม
- copy dataset ไป local SSD ถ้าพื้นที่เพียงพอ

## Training mode

```text
auto        = มี checkpoint ให้ resume, ไม่มีจึงเริ่มจาก Hugging Face
resume      = บังคับ resume จาก checkpoint
pretrained  = เริ่ม model ใหม่จาก TerraMind pretrained บน Hugging Face
```

## เมื่อต้องใช้ dataset ใหม่

แก้เพียง:

```python
PROJECT_ROOT
DATASET_DIR
MODEL_DIR
```

และตรวจให้ HDF5 เป็นไปตาม Dataset contract

In [ ]:
# ============================================================
# 04 — USER CONFIGURATION
# ============================================================

# ---------- Shared project root ----------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Rice_Detect_TerraMind"
)

# ---------- Input dataset ----------
DATASET_DIR = (
    PROJECT_ROOT
    / "AOI_output"
    / "processed_2026"
    / "rice_train_ready_multitemporal_final"
)

# ---------- Model/checkpoint/config directory ----------
MODEL_DIR = (
    PROJECT_ROOT
    / "Rice_TerraMind_Training"
    / "terramind_rice_multitemporal_final"
)

# ---------- Output ----------
# เก็บ canonical best.ckpt / last.ckpt และไฟล์ config กลับ MODEL_DIR
OUTPUT_DIR = MODEL_DIR

RUN_NAME = "terramind_rice_showcase_resume"

# ---------- Training behavior ----------
TRAINING_MODE = "auto"       # "auto", "resume", "pretrained"
RESUME_CHECKPOINT_NAME = "last.ckpt"
ADDITIONAL_EPOCHS = 3        # showcase: ต่อจาก checkpoint อีก 2–3 epochs
NEW_TRAINING_EPOCHS = 3      # ใช้เมื่อ TRAINING_MODE="pretrained"

# ---------- Data ----------
CACHE_DATASET_TO_LOCAL = True
LOCAL_DATASET_DIR = Path("/content/rice_dataset")
REUSE_EXISTING_NORMALIZATION = True
REUSE_EXISTING_CLASS_WEIGHTS = True
NORMALIZATION_PIXEL_STEP = 8

TEMPORAL_SELECTION = "all"   # "all" หรือ "top_quality"
TOP_QUALITY_TIMESTAMPS = 4

# ---------- Optimization ----------
BASE_LEARNING_RATE = 1e-4
BACKBONE_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 6

# ---------- Evaluation ----------
CALIBRATE_THRESHOLD = True
THRESHOLD_CANDIDATES = [
    0.30, 0.35, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.65, 0.70,
]
SHOW_SAMPLE_PREVIEW = True

# ---------- Hugging Face ----------
# Public model ใช้ได้โดยไม่ต้องมี token แต่ token ช่วยเพิ่ม rate limit
HF_REPO_ID = "ibm-esa-geospatial/TerraMind-1.0-base"
HF_FILENAME = "TerraMind_v1_base.pt"
HF_CACHE_ROOT = PROJECT_ROOT / "_huggingface_cache"
HF_DOWNLOAD_RETRIES = 5

# ---------- Advanced ----------
STRICT_DATASET_CHECK = True
KEEP_HISTORY_CHECKPOINTS = False


print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Model directory:", MODEL_DIR)
print("Training mode:", TRAINING_MODE)
print("Additional epochs:", ADDITIONAL_EPOCHS)

# 4 — ตรวจ path และเลือก checkpoint

หลักการเลือก checkpoint สำหรับ resume:

1. ใช้ชื่อที่กำหนดใน `RESUME_CHECKPOINT_NAME`
2. ถ้าไม่พบ ให้ลอง `last.ckpt`
3. ถ้ายังไม่พบ ให้เลือก `best-rice-iou-XX.ckpt` ที่ epoch สูงสุด
4. ถ้า `TRAINING_MODE="auto"` และไม่พบ checkpoint จะเริ่มจาก pretrained TerraMind
5. ถ้า `TRAINING_MODE="resume"` แต่ไม่พบ checkpoint จะหยุดพร้อมข้อความชัดเจน

สำหรับการ resume ควรใช้ `last.ckpt` เพราะเก็บ training state ล่าสุด รวม optimizer และ scheduler

In [ ]:
# ============================================================
# 05 — PATH VALIDATION + CHECKPOINT SELECTION
# ============================================================

for required_path in [PROJECT_ROOT, DATASET_DIR, MODEL_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"ไม่พบ path:\n{required_path}\n"
            "ตรวจชื่อ shortcut ใน My Drive และ PROJECT_ROOT"
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def extract_epoch(path: Path) -> int:
    patterns = [
        r"epoch[=_-]?(\d+)",
        r"rice-iou-(\d+)",
        r"loss-(\d+)",
        r"(\d+)(?=\.ckpt$)",
    ]

    for pattern in patterns:
        match = re.search(pattern, path.name, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))

    return -1


def select_resume_checkpoint(model_dir: Path) -> Path | None:
    preferred = model_dir / RESUME_CHECKPOINT_NAME
    if preferred.exists():
        return preferred

    last_path = model_dir / "last.ckpt"
    if last_path.exists():
        return last_path

    candidates = sorted(
        model_dir.glob("*rice*iou*.ckpt"),
        key=extract_epoch,
    )
    return candidates[-1] if candidates else None


RESUME_CHECKPOINT_PATH = select_resume_checkpoint(MODEL_DIR)

if TRAINING_MODE not in {"auto", "resume", "pretrained"}:
    raise ValueError(
        "TRAINING_MODE ต้องเป็น auto, resume หรือ pretrained"
    )

if TRAINING_MODE == "resume" and RESUME_CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        f"TRAINING_MODE='resume' แต่ไม่พบ checkpoint ใน {MODEL_DIR}"
    )

USE_RESUME = (
    TRAINING_MODE == "resume"
    or (
        TRAINING_MODE == "auto"
        and RESUME_CHECKPOINT_PATH is not None
    )
)

print("Resume enabled:", USE_RESUME)
print("Resume checkpoint:", RESUME_CHECKPOINT_PATH)

# 5 — Copy dataset ไป local SSD อย่างปลอดภัย

HDF5 ที่อ่านผ่าน Google Drive mount อาจช้า โดยเฉพาะเมื่อ DataLoader ขอ sample จำนวนมาก

Cell นี้:

- คำนวณขนาด dataset
- ตรวจพื้นที่ว่างใน `/content`
- ใช้ `rsync` เมื่อพื้นที่เพียงพอ
- ตรวจจำนวน byte หลัง copy
- fallback ไปอ่านจาก Drive โดยตรงเมื่อ local disk ไม่พอ

การ copy ไม่ได้เปลี่ยนไฟล์ต้นฉบับ

In [ ]:
# ============================================================
# 06 — LOCAL DATASET CACHE
# ============================================================

def directory_size_bytes(path: Path) -> int:
    return sum(
        item.stat().st_size
        for item in path.rglob("*")
        if item.is_file()
    )


dataset_bytes = directory_size_bytes(DATASET_DIR)
free_bytes = shutil.disk_usage("/content").free
safety_margin_bytes = 6 * 1024**3

print("Dataset size:", f"{dataset_bytes / 1024**3:.2f} GB")
print("Local free space:", f"{free_bytes / 1024**3:.2f} GB")


USE_LOCAL_DATASET = (
    CACHE_DATASET_TO_LOCAL
    and free_bytes > dataset_bytes + safety_margin_bytes
)

if USE_LOCAL_DATASET:
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

    command = [
        "rsync",
        "-ah",
        "--partial",
        "--info=progress2",
        str(DATASET_DIR) + "/",
        str(LOCAL_DATASET_DIR) + "/",
    ]

    subprocess.run(command, check=True)

    copied_bytes = directory_size_bytes(LOCAL_DATASET_DIR)

    if copied_bytes != dataset_bytes:
        raise RuntimeError(
            "Dataset copy size mismatch\n"
            f"Source: {dataset_bytes}\n"
            f"Local: {copied_bytes}"
        )

    ACTIVE_DATASET_DIR = LOCAL_DATASET_DIR
else:
    ACTIVE_DATASET_DIR = DATASET_DIR

print("Using local dataset:", USE_LOCAL_DATASET)
print("Active dataset:", ACTIVE_DATASET_DIR)

# 6 — ค้นหาและตรวจ HDF5 shards

รองรับสองโครงสร้าง:

```text
dataset/shards/train_shard_000.h5
```

หรือ:

```text
dataset/train_shard_000.h5
```

ระบบจะ:

- ค้นหา train/val/test
- ตรวจ datasets ที่จำเป็น
- ตรวจ shape `[N,T,C,H,W]`
- ตรวจ label values
- อ่าน timestamps และ band names
- ไม่โหลดภาพทั้ง dataset เข้า RAM

In [ ]:
# ============================================================
# 07 — HDF5 DISCOVERY + FAST VALIDATION
# ============================================================

SHARD_DIR = (
    ACTIVE_DATASET_DIR / "shards"
    if (ACTIVE_DATASET_DIR / "shards").exists()
    else ACTIVE_DATASET_DIR
)


def decode_text(value) -> str:
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


def find_split_shards(split: str) -> list[Path]:
    patterns = [
        f"{split}_shard_*.h5",
        f"{split}*.h5",
    ]

    found = []
    for pattern in patterns:
        found.extend(SHARD_DIR.glob(pattern))

    return sorted(set(found))


split_shards = {
    split: find_split_shards(split)
    for split in ["train", "val", "test"]
}

for split, shard_paths in split_shards.items():
    if not shard_paths:
        raise FileNotFoundError(
            f"ไม่พบ {split} shards ใน {SHARD_DIR}"
        )


def inspect_shard(path: Path, inspect_label_sample: bool = True) -> dict:
    with h5py.File(path, "r") as file:
        required = {
            "images",
            "valid_masks",
            "timestep_quality",
            "labels",
        }

        missing = required - set(file.keys())
        if missing:
            raise RuntimeError(
                f"{path.name} missing datasets: {sorted(missing)}"
            )

        images = file["images"]
        valid_masks = file["valid_masks"]
        quality = file["timestep_quality"]
        labels = file["labels"]

        if images.ndim != 5:
            raise RuntimeError(
                f"{path.name}: images ต้องเป็น [N,T,C,H,W], พบ {images.shape}"
            )

        expected_label_shape = (
            images.shape[0],
            images.shape[-2],
            images.shape[-1],
        )

        if labels.shape != expected_label_shape:
            raise RuntimeError(
                f"{path.name}: image-label shape mismatch"
            )

        if valid_masks.shape != (
            images.shape[0],
            images.shape[1],
            images.shape[-2],
            images.shape[-1],
        ):
            raise RuntimeError(
                f"{path.name}: valid_masks shape mismatch"
            )

        if quality.shape != (images.shape[0], images.shape[1]):
            raise RuntimeError(
                f"{path.name}: timestep_quality shape mismatch"
            )

        if inspect_label_sample:
            values = set(np.unique(labels[0]).tolist())
            if not values.issubset({0, 1, 255}):
                raise RuntimeError(
                    f"{path.name}: unexpected labels {sorted(values)}"
                )

        timestamps = (
            [decode_text(item) for item in file["timestamps"][:]]
            if "timestamps" in file
            else [str(index) for index in range(images.shape[1])]
        )

        band_names = (
            [decode_text(item) for item in file["band_names"][:]]
            if "band_names" in file
            else [f"BAND_{index + 1}" for index in range(images.shape[2])]
        )

        return {
            "samples": int(images.shape[0]),
            "sample_shape": tuple(images.shape[1:]),
            "timestamps": timestamps,
            "band_names": band_names,
            "dtype": str(images.dtype),
        }


reference_info = inspect_shard(split_shards["train"][0])

if STRICT_DATASET_CHECK:
    for split, shard_paths in split_shards.items():
        for shard_path in tqdm(
            shard_paths,
            desc=f"Checking {split} shards",
        ):
            info = inspect_shard(
                shard_path,
                inspect_label_sample=False,
            )

            if info["sample_shape"] != reference_info["sample_shape"]:
                raise RuntimeError(
                    f"Inconsistent sample shape: {shard_path.name}"
                )

            if info["timestamps"] != reference_info["timestamps"]:
                raise RuntimeError(
                    f"Inconsistent timestamps: {shard_path.name}"
                )

            if info["band_names"] != reference_info["band_names"]:
                raise RuntimeError(
                    f"Inconsistent band names: {shard_path.name}"
                )


def count_samples(shard_paths: list[Path]) -> int:
    total = 0
    for shard_path in shard_paths:
        with h5py.File(shard_path, "r") as file:
            total += int(file["images"].shape[0])
    return total


split_counts = {
    split: count_samples(shard_paths)
    for split, shard_paths in split_shards.items()
}

print("Dataset ready")
print("Sample shape [T,C,H,W]:", reference_info["sample_shape"])
print("Timestamps:", reference_info["timestamps"])
print("Bands:", reference_info["band_names"])

for split, count in split_counts.items():
    print(f"{split:5s}: {count:,} samples")

# 7 — ตรวจ compatibility ระหว่าง dataset และ checkpoint

Resume ทำได้อย่างปลอดภัยเมื่อ architecture และ input contract ตรงกัน โดยเฉพาะ:

- จำนวน bands
- band order
- timestamps
- number of classes
- temporal configuration

หาก dataset ใหม่ไม่ตรงกับ checkpoint เดิม Notebook จะหยุด และแนะนำให้ใช้:

```python
TRAINING_MODE = "pretrained"
```

แทนการฝืน resume

In [ ]:
# ============================================================
# 08 — LOAD EXISTING CONFIG + COMPATIBILITY CHECK
# ============================================================

TRAINING_CONFIG_PATH = MODEL_DIR / "training_config.json"
NORMALIZATION_PATH = MODEL_DIR / "normalization_stats.json"

existing_training_config = None
existing_normalization = None

if TRAINING_CONFIG_PATH.exists():
    with open(TRAINING_CONFIG_PATH, "r", encoding="utf-8") as file:
        existing_training_config = json.load(file)

if NORMALIZATION_PATH.exists():
    with open(NORMALIZATION_PATH, "r", encoding="utf-8") as file:
        existing_normalization = json.load(file)


DATASET_TIMESTAMPS = reference_info["timestamps"]
DATASET_BANDS = reference_info["band_names"]


if USE_RESUME and existing_training_config is not None:
    checkpoint_timestamps = [
        str(value)
        for value in existing_training_config.get("timestamps", [])
    ]

    checkpoint_bands = existing_training_config.get(
        "bands",
        existing_training_config.get("normalization", {}).get(
            "band_names",
            [],
        ),
    )

    if checkpoint_timestamps and checkpoint_timestamps != DATASET_TIMESTAMPS:
        raise RuntimeError(
            "Dataset timestamps ไม่ตรงกับ checkpoint\n"
            f"Dataset: {DATASET_TIMESTAMPS}\n"
            f"Checkpoint: {checkpoint_timestamps}\n"
            "เปลี่ยน TRAINING_MODE='pretrained' สำหรับ dataset ใหม่"
        )

    if checkpoint_bands and list(checkpoint_bands) != DATASET_BANDS:
        raise RuntimeError(
            "Dataset bands ไม่ตรงกับ checkpoint\n"
            f"Dataset: {DATASET_BANDS}\n"
            f"Checkpoint: {checkpoint_bands}\n"
            "เปลี่ยน TRAINING_MODE='pretrained' สำหรับ dataset ใหม่"
        )

print("Dataset-checkpoint compatibility: PASSED")

# 8 — Normalization และ class weights

ลำดับการเลือก normalization:

1. เมื่อ resume และมี `normalization_stats.json` ให้ใช้ค่าเดิม
2. เมื่อ dataset มีไฟล์ statistics ใน `configs/` ให้ใช้ไฟล์นั้น
3. หากไม่มี ให้คำนวณจาก train split

สำหรับ dataset ใหม่ ไม่ควรใช้ normalization ของ dataset เก่า

Class weight ใช้ square-root imbalance ratio และจำกัดไว้ไม่เกิน 4 เพื่อลดความรุนแรงของ weight

In [ ]:
# ============================================================
# 09 — NORMALIZATION + CLASS WEIGHTS
# ============================================================

def find_dataset_normalization() -> Path | None:
    config_dir = ACTIVE_DATASET_DIR / "configs"
    candidates = []

    if config_dir.exists():
        candidates.extend(config_dir.glob("normalization_stats*.json"))

    candidates.extend(ACTIVE_DATASET_DIR.glob("normalization_stats*.json"))

    return sorted(candidates)[0] if candidates else None


def compute_normalization_stats(
    shard_paths: list[Path],
    pixel_step: int,
) -> dict:
    band_sum = None
    band_sum_sq = None
    band_count = None

    for shard_path in tqdm(
        shard_paths,
        desc="Computing normalization",
    ):
        with h5py.File(shard_path, "r") as file:
            images_ds = file["images"]
            valid_ds = file["valid_masks"]

            if band_sum is None:
                n_bands = images_ds.shape[2]
                band_sum = np.zeros(n_bands, dtype=np.float64)
                band_sum_sq = np.zeros(n_bands, dtype=np.float64)
                band_count = np.zeros(n_bands, dtype=np.int64)

            for sample_index in range(images_ds.shape[0]):
                images = images_ds[sample_index][
                    :, :, ::pixel_step, ::pixel_step
                ].astype(np.float64)

                valid = valid_ds[sample_index][
                    :, ::pixel_step, ::pixel_step
                ].astype(bool)

                for band_index in range(images.shape[1]):
                    values = images[:, band_index][valid]

                    if values.size == 0:
                        continue

                    band_sum[band_index] += values.sum()
                    band_sum_sq[band_index] += np.square(values).sum()
                    band_count[band_index] += values.size

    means = band_sum / np.maximum(band_count, 1)

    variance = (
        band_sum_sq / np.maximum(band_count, 1)
        - np.square(means)
    )

    stds = np.sqrt(np.maximum(variance, 1.0))

    return {
        "band_names": DATASET_BANDS,
        "means": means.tolist(),
        "stds": stds.tolist(),
        "pixel_step": pixel_step,
        "valid_pixel_counts": band_count.tolist(),
    }


dataset_normalization_path = find_dataset_normalization()

if (
    USE_RESUME
    and REUSE_EXISTING_NORMALIZATION
    and existing_normalization is not None
):
    normalization_stats = existing_normalization
    normalization_source = NORMALIZATION_PATH

elif (
    dataset_normalization_path is not None
    and REUSE_EXISTING_NORMALIZATION
):
    with open(
        dataset_normalization_path,
        "r",
        encoding="utf-8",
    ) as file:
        normalization_stats = json.load(file)

    normalization_source = dataset_normalization_path

else:
    normalization_stats = compute_normalization_stats(
        split_shards["train"],
        pixel_step=NORMALIZATION_PIXEL_STEP,
    )
    normalization_source = "computed"


BAND_NAMES = list(normalization_stats["band_names"])
BAND_MEANS = np.asarray(
    normalization_stats["means"],
    dtype=np.float32,
)
BAND_STDS = np.asarray(
    normalization_stats["stds"],
    dtype=np.float32,
)

if BAND_NAMES != DATASET_BANDS:
    raise RuntimeError(
        "Normalization band order ไม่ตรงกับ dataset"
    )

if BAND_MEANS.shape != (len(DATASET_BANDS),):
    raise RuntimeError("Normalization means shape mismatch")

if BAND_STDS.shape != (len(DATASET_BANDS),):
    raise RuntimeError("Normalization stds shape mismatch")


def count_training_pixels(shard_paths: list[Path]) -> dict[int, int]:
    counts = {0: 0, 1: 0, 255: 0}

    for shard_path in tqdm(
        shard_paths,
        desc="Counting label pixels",
    ):
        with h5py.File(shard_path, "r") as file:
            labels = file["labels"]

            for sample_index in range(labels.shape[0]):
                label = labels[sample_index]

                counts[0] += int(np.count_nonzero(label == 0))
                counts[1] += int(np.count_nonzero(label == 1))
                counts[255] += int(np.count_nonzero(label == 255))

    return counts


existing_class_weights = (
    existing_training_config.get("class_weights")
    if existing_training_config
    else None
)

if (
    USE_RESUME
    and REUSE_EXISTING_CLASS_WEIGHTS
    and existing_class_weights is not None
):
    CLASS_WEIGHTS = [
        float(value)
        for value in existing_class_weights
    ]
    pixel_counts = None

else:
    pixel_counts = count_training_pixels(split_shards["train"])

    non_rice_pixels = pixel_counts[0]
    rice_pixels = pixel_counts[1]

    rice_weight = math.sqrt(
        non_rice_pixels / max(rice_pixels, 1)
    )

    rice_weight = float(
        np.clip(rice_weight, 1.0, 4.0)
    )

    CLASS_WEIGHTS = [1.0, rice_weight]


display(pd.DataFrame({
    "band": BAND_NAMES,
    "mean": BAND_MEANS,
    "std": BAND_STDS,
}))

print("Normalization source:", normalization_source)
print("Class weights:", CLASS_WEIGHTS)

# 9 — Dataset และ DataModule

`RiceH5Dataset`:

- เปิด HDF5 เฉพาะเมื่อ worker ต้องใช้
- cache file handle แยกตาม worker
- เลือก timestamps ได้
- ใช้ augmentation เดียวกันทุก timestamp
- ใช้ spectral gain แบบอ่อนต่อ band
- transpose `[T,C,H,W] → [C,T,H,W]`
- normalize ด้วยค่าเดียวกับ prediction

`DataModule` ทำให้ train/val/test ใช้ contract เดียวกัน

In [ ]:
# ============================================================
# 10 — PYTORCH DATASET + DATAMODULE
# ============================================================

class RiceH5Dataset(Dataset):
    def __init__(
        self,
        shard_paths: list[Path],
        means: np.ndarray,
        stds: np.ndarray,
        train: bool,
        temporal_selection: str,
        top_quality_timestamps: int,
    ):
        self.shard_paths = [Path(path) for path in shard_paths]
        self.means = np.asarray(means, dtype=np.float32)
        self.stds = np.asarray(stds, dtype=np.float32)
        self.train = bool(train)
        self.temporal_selection = temporal_selection
        self.top_quality_timestamps = int(top_quality_timestamps)

        self.samples = []
        self._handles = {}

        for shard_path in self.shard_paths:
            with h5py.File(shard_path, "r") as file:
                sample_count = int(file["images"].shape[0])

            self.samples.extend(
                (str(shard_path), local_index)
                for local_index in range(sample_count)
            )

    def __len__(self):
        return len(self.samples)

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_handles"] = {}
        return state

    def _get_handle(self, path: str):
        if path not in self._handles:
            self._handles[path] = h5py.File(path, "r")
        return self._handles[path]

    def _select_timestamps(
        self,
        images: np.ndarray,
        quality: np.ndarray,
    ) -> np.ndarray:
        if self.temporal_selection == "all":
            return images

        if self.temporal_selection == "top_quality":
            keep_count = min(
                self.top_quality_timestamps,
                images.shape[0],
            )

            indices = np.argsort(quality)[-keep_count:]
            indices = np.sort(indices)

            return images[indices]

        raise ValueError(
            "temporal_selection ต้องเป็น all หรือ top_quality"
        )

    @staticmethod
    def _spatial_augmentation(
        image: np.ndarray,
        mask: np.ndarray,
    ):
        if random.random() < 0.5:
            image = np.flip(image, axis=-1)
            mask = np.flip(mask, axis=-1)

        if random.random() < 0.5:
            image = np.flip(image, axis=-2)
            mask = np.flip(mask, axis=-2)

        rotation_k = random.randint(0, 3)

        if rotation_k:
            image = np.rot90(
                image,
                k=rotation_k,
                axes=(-2, -1),
            )
            mask = np.rot90(
                mask,
                k=rotation_k,
                axes=(-2, -1),
            )

        return (
            np.ascontiguousarray(image),
            np.ascontiguousarray(mask),
        )

    def __getitem__(self, index: int):
        shard_path, local_index = self.samples[index]
        file = self._get_handle(shard_path)

        image = file["images"][local_index].astype(np.float32)
        quality = file["timestep_quality"][local_index].astype(np.float32)
        mask = file["labels"][local_index].astype(np.int64)

        image = self._select_timestamps(image, quality)

        if self.train:
            image, mask = self._spatial_augmentation(image, mask)

            spectral_gain = np.random.uniform(
                0.97,
                1.03,
                size=(1, image.shape[1], 1, 1),
            ).astype(np.float32)

            image = image * spectral_gain

        # [T,C,H,W] -> [C,T,H,W]
        image = np.transpose(image, (1, 0, 2, 3))

        image = (
            image
            - self.means[:, None, None, None]
        ) / self.stds[:, None, None, None]

        return {
            "image": torch.from_numpy(
                np.ascontiguousarray(image)
            ).float(),
            "mask": torch.from_numpy(
                np.ascontiguousarray(mask)
            ).long(),
            "filename": f"{Path(shard_path).name}:{local_index}",
        }


if GPU_MEMORY_GB >= 70:
    BATCH_SIZE = 8
    ACCUMULATE_GRAD_BATCHES = 1
    NUM_WORKERS = 6
elif GPU_MEMORY_GB >= 38:
    BATCH_SIZE = 4
    ACCUMULATE_GRAD_BATCHES = 2
    NUM_WORKERS = 4
elif GPU_MEMORY_GB >= 20:
    BATCH_SIZE = 2
    ACCUMULATE_GRAD_BATCHES = 4
    NUM_WORKERS = 2
else:
    BATCH_SIZE = 1
    ACCUMULATE_GRAD_BATCHES = 8
    NUM_WORKERS = 2


TRAIN_PRECISION = (
    "bf16-mixed"
    if GPU_CAPABILITY[0] >= 8
    else "16-mixed"
)


class RiceH5DataModule(pl.LightningDataModule):
    def __init__(self):
        super().__init__()

    def setup(self, stage=None):
        common = {
            "means": BAND_MEANS,
            "stds": BAND_STDS,
            "temporal_selection": TEMPORAL_SELECTION,
            "top_quality_timestamps": TOP_QUALITY_TIMESTAMPS,
        }

        if stage in (None, "fit", "validate"):
            self.train_dataset = RiceH5Dataset(
                split_shards["train"],
                train=True,
                **common,
            )
            self.val_dataset = RiceH5Dataset(
                split_shards["val"],
                train=False,
                **common,
            )

        if stage in (None, "test", "predict"):
            self.test_dataset = RiceH5Dataset(
                split_shards["test"],
                train=False,
                **common,
            )

    def _loader(self, dataset, shuffle: bool, drop_last: bool):
        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            persistent_workers=NUM_WORKERS > 0,
            drop_last=drop_last,
        )

    def train_dataloader(self):
        return self._loader(
            self.train_dataset,
            shuffle=True,
            drop_last=True,
        )

    def val_dataloader(self):
        return self._loader(
            self.val_dataset,
            shuffle=False,
            drop_last=False,
        )

    def test_dataloader(self):
        return self._loader(
            self.test_dataset,
            shuffle=False,
            drop_last=False,
        )


data_module = RiceH5DataModule()
data_module.setup("fit")

print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", ACCUMULATE_GRAD_BATCHES)
print("Effective batch:", BATCH_SIZE * ACCUMULATE_GRAD_BATCHES)
print("Workers:", NUM_WORKERS)
print("Precision:", TRAIN_PRECISION)
print("Train samples:", len(data_module.train_dataset))
print("Validation samples:", len(data_module.val_dataset))

# 10 — กำหนดจำนวน epoch

Lightning ใช้ `max_epochs` เป็นจำนวน epoch รวมทั้งหมด ไม่ใช่จำนวน epoch ที่ต้องการเพิ่ม

ตัวอย่าง:

```text
checkpoint epoch = 28
ADDITIONAL_EPOCHS = 3
MAX_EPOCHS_TOTAL = 32
```

เพราะ epoch เริ่มนับจาก 0 และ checkpoint epoch 28 หมายถึง train จบแล้ว 29 epochs

In [ ]:
# ============================================================
# 11 — EPOCH PLAN
# ============================================================

RESUME_EPOCH = -1

if USE_RESUME:
    checkpoint_metadata = torch.load(
        RESUME_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

    RESUME_EPOCH = int(
        checkpoint_metadata.get("epoch", -1)
    )

    if "state_dict" not in checkpoint_metadata:
        raise RuntimeError("Checkpoint ไม่มี state_dict")

    del checkpoint_metadata
    gc.collect()

    MAX_EPOCHS_TOTAL = (
        RESUME_EPOCH + 1 + ADDITIONAL_EPOCHS
    )
else:
    MAX_EPOCHS_TOTAL = NEW_TRAINING_EPOCHS


print("Resume epoch:", RESUME_EPOCH)
print("Total max epochs:", MAX_EPOCHS_TOTAL)
print(
    "Epochs to run:",
    ADDITIONAL_EPOCHS if USE_RESUME else NEW_TRAINING_EPOCHS,
)

# 11 — Hugging Face pretrained cache

ใช้เฉพาะเมื่อเริ่มจาก pretrained TerraMind

ระบบจะ:

- ใช้ public Hugging Face model
- อ่าน `HF_TOKEN` จาก Colab Secrets หากมี
- cache ไฟล์ไว้ใน project folder
- retry เมื่อเกิด HTTP 504 หรือ network timeout
- เมื่อไฟล์อยู่ใน cache แล้ว runtime ครั้งถัดไปไม่ต้องดาวน์โหลดใหม่

การ resume จาก checkpoint ไม่จำเป็นต้องดาวน์โหลด pretrained weight ซ้ำ

In [ ]:
# ============================================================
# 12 — OPTIONAL HUGGING FACE CACHE
# ============================================================

HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")


def ensure_terramind_pretrained_cached() -> Path:
    last_error = None

    for attempt in range(1, HF_DOWNLOAD_RETRIES + 1):
        try:
            path = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=HF_FILENAME,
                cache_dir=os.environ["HF_HUB_CACHE"],
                token=HF_TOKEN,
                force_download=False,
            )

            print("TerraMind pretrained cache:", path)
            return Path(path)

        except Exception as error:
            last_error = error
            wait_seconds = min(2 ** attempt, 30)

            print(
                f"HF attempt {attempt}/{HF_DOWNLOAD_RETRIES} failed:",
                type(error).__name__,
            )

            if attempt < HF_DOWNLOAD_RETRIES:
                time.sleep(wait_seconds)

    raise RuntimeError(
        "ดาวน์โหลด TerraMind pretrained ไม่สำเร็จ\n"
        "ตรวจ internet, HF_TOKEN หรือรันใหม่ภายหลัง"
    ) from last_error


if not USE_RESUME:
    PRETRAINED_CACHE_PATH = ensure_terramind_pretrained_cached()
else:
    PRETRAINED_CACHE_PATH = None
    print("Resume mode: ไม่ต้องดาวน์โหลด pretrained ซ้ำ")

# 12 — สร้าง TerraMind semantic segmentation task

Architecture ที่ใช้ใน showcase:

- Backbone: TerraMind v1 Base
- Modality: Sentinel-2 L2A
- Bands: Blue, Green, Red, Narrow NIR, SWIR1, SWIR2
- Temporal pooling: mean
- Decoder: U-Net
- Loss: 60% Cross Entropy + 40% Dice
- Encoder learning rate ต่ำกว่า decoder

## Resume mode

สร้าง architecture โดยตั้ง `backbone_pretrained=False` เพราะน้ำหนักทั้งหมดจะถูก restore จาก Lightning checkpoint

## Pretrained mode

ตั้ง `backbone_pretrained=True` เพื่อเริ่มจาก TerraMind ที่ดาวน์โหลดจาก Hugging Face

In [ ]:
# ============================================================
# 13 — BUILD TERRAMIND TASK
# ============================================================

DEFAULT_MODEL_ARGS = {
    "backbone": "terramind_v1_base",
    "backbone_pretrained": not USE_RESUME,
    "backbone_modalities": ["S2L2A"],
    "backbone_bands": {
        "S2L2A": [
            "BLUE",
            "GREEN",
            "RED",
            "NIR_NARROW",
            "SWIR_1",
            "SWIR_2",
        ]
    },
    "backbone_use_temporal": True,
    "backbone_temporal_pooling": "mean",
    "necks": [
        {
            "name": "SelectIndices",
            "indices": [2, 5, 8, 11],
        },
        {
            "name": "ReshapeTokensToImage",
            "remove_cls_token": False,
        },
        {
            "name": "LearnedInterpolateToPyramidal",
        },
    ],
    "decoder": "UNetDecoder",
    "decoder_channels": [512, 256, 128, 64],
    "head_dropout": 0.15,
    "num_classes": 2,
}


if USE_RESUME and existing_training_config is not None:
    MODEL_ARGS = copy.deepcopy(
        existing_training_config.get(
            "model_args",
            DEFAULT_MODEL_ARGS,
        )
    )

    # ใช้เฉพาะ key ที่ EncoderDecoderFactory รองรับ
    if "backbone_pretrained" in MODEL_ARGS:
        MODEL_ARGS["backbone_pretrained"] = False
else:
    MODEL_ARGS = copy.deepcopy(DEFAULT_MODEL_ARGS)


model = terratorch.tasks.SemanticSegmentationTask(
    model_factory="EncoderDecoderFactory",
    model_args=MODEL_ARGS,
    loss={
        "ce": 0.60,
        "dice": 0.40,
    },
    class_weights=CLASS_WEIGHTS,
    ignore_index=255,
    lr=BASE_LEARNING_RATE,
    optimizer="AdamW",
    optimizer_hparams={
        "weight_decay": WEIGHT_DECAY,
        "betas": (0.9, 0.999),
    },
    scheduler="CosineAnnealingLR",
    scheduler_hparams={
        "T_max": max(MAX_EPOCHS_TOTAL, 1),
        "eta_min": 1e-6,
    },
    freeze_backbone=False,
    freeze_decoder=False,
    lr_overrides={
        "model.encoder": BACKBONE_LEARNING_RATE,
    },
    plot_on_val=False,
    class_names=[
        "non_rice",
        "rice",
    ],
)

print("TerraMind task created")
print("Backbone pretrained initialization:", MODEL_ARGS.get("backbone_pretrained"))

# 13 — Smoke test ก่อน train

Smoke test ใช้เพียงหนึ่ง batch เพื่อตรวจ:

- DataLoader อ่าน HDF5 ได้
- input shape เป็น `[B,C,T,H,W]`
- model forward ผ่าน
- output มี 2 classes
- spatial size ตรงกับ label
- output ไม่มี NaN หรือ Inf

หลังจบ test จะเรียก `model.train()` เพื่อไม่ให้โมเดลค้างใน eval mode

In [ ]:
# ============================================================
# 14 — SMOKE TEST
# ============================================================

train_batch = next(iter(data_module.train_dataloader()))

if not hasattr(model, "model"):
    model.configure_models()

model = model.to(DEVICE)
model.eval()

smoke_image = train_batch["image"].to(
    DEVICE,
    non_blocking=True,
)

amp_dtype = (
    torch.bfloat16
    if TRAIN_PRECISION == "bf16-mixed"
    else torch.float16
)

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        dtype=amp_dtype,
    ):
        smoke_result = model(smoke_image)
        smoke_output = (
            smoke_result.output
            if hasattr(smoke_result, "output")
            else smoke_result
        )


if smoke_output.shape[0] != smoke_image.shape[0]:
    raise RuntimeError("Batch dimension mismatch")

if smoke_output.shape[1] != 2:
    raise RuntimeError("Expected 2 output classes")

if smoke_output.shape[-2:] != smoke_image.shape[-2:]:
    raise RuntimeError("Spatial output size mismatch")

if not torch.isfinite(smoke_output).all():
    raise RuntimeError("Model output contains NaN or Inf")


print("Input shape:", tuple(smoke_image.shape))
print("Output shape:", tuple(smoke_output.shape))
print("Smoke test: PASSED")


model.train()

del (
    smoke_image,
    smoke_output,
    smoke_result,
    train_batch,
)

gc.collect()
torch.cuda.empty_cache()

# 14 — Checkpoint, logging และ backup

Training ทำงานบน local Colab เพื่อความเร็ว แต่สำรองไป Google Drive ทุก validation epoch

ไฟล์หลัก:

```text
best.ckpt                 checkpoint ที่ Rice IoU ดีที่สุดใน run นี้
last.ckpt                 training state ล่าสุด ใช้ resume
training_config.json      architecture และ hyperparameters
normalization_stats.json  mean/std ที่ prediction ต้องใช้
training_history.csv      metrics ราย epoch
```

Historical checkpoints ปิดไว้เป็น default เพื่อลดความสับสนและพื้นที่

In [ ]:
# ============================================================
# 15 — OUTPUT + CALLBACKS
# ============================================================

LOCAL_RUN_DIR = Path("/content/rice_training") / RUN_NAME
LOCAL_CHECKPOINT_DIR = LOCAL_RUN_DIR / "checkpoints"
LOCAL_LOG_DIR = LOCAL_RUN_DIR / "logs"
LOCAL_CONFIG_DIR = LOCAL_RUN_DIR / "configs"

for directory in [
    LOCAL_RUN_DIR,
    LOCAL_CHECKPOINT_DIR,
    LOCAL_LOG_DIR,
    LOCAL_CONFIG_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


best_rice_callback = ModelCheckpoint(
    dirpath=LOCAL_CHECKPOINT_DIR,
    filename="best-rice-iou-{epoch:02d}",
    monitor="val/IoU_rice",
    mode="max",
    save_top_k=1,
    save_last=True,
    auto_insert_metric_name=False,
)

early_stopping = EarlyStopping(
    monitor="val/IoU_rice",
    mode="max",
    patience=EARLY_STOP_PATIENCE,
    min_delta=0.001,
    verbose=True,
)

lr_monitor = LearningRateMonitor(
    logging_interval="epoch"
)


class CanonicalDriveBackup(Callback):
    def __init__(
        self,
        checkpoint_callback: ModelCheckpoint,
        local_checkpoint_dir: Path,
        drive_output_dir: Path,
    ):
        super().__init__()
        self.checkpoint_callback = checkpoint_callback
        self.local_checkpoint_dir = Path(local_checkpoint_dir)
        self.drive_output_dir = Path(drive_output_dir)

    def _copy_if_exists(self, source: Path, destination_name: str):
        if source.exists():
            temporary = (
                self.drive_output_dir
                / f"{destination_name}.partial"
            )

            shutil.copy2(source, temporary)
            temporary.replace(
                self.drive_output_dir / destination_name
            )

    def backup(self):
        self.drive_output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        best_path = Path(
            self.checkpoint_callback.best_model_path
        ) if self.checkpoint_callback.best_model_path else None

        last_path = self.local_checkpoint_dir / "last.ckpt"

        if best_path is not None:
            self._copy_if_exists(best_path, "best.ckpt")

            if KEEP_HISTORY_CHECKPOINTS:
                self._copy_if_exists(
                    best_path,
                    best_path.name,
                )

        self._copy_if_exists(last_path, "last.ckpt")

        for config_file in LOCAL_CONFIG_DIR.glob("*"):
            if config_file.is_file():
                self._copy_if_exists(
                    config_file,
                    config_file.name,
                )

    def on_validation_end(self, trainer, pl_module):
        self.backup()

    def on_train_end(self, trainer, pl_module):
        self.backup()


drive_backup_callback = CanonicalDriveBackup(
    checkpoint_callback=best_rice_callback,
    local_checkpoint_dir=LOCAL_CHECKPOINT_DIR,
    drive_output_dir=OUTPUT_DIR,
)

print("Local run:", LOCAL_RUN_DIR)
print("Drive output:", OUTPUT_DIR)

# 15 — บันทึก configuration ก่อน train

Configuration ทำให้ Prediction Notebook รู้ว่า:

- model architecture คืออะไร
- ใช้ bands และ timestamps ใด
- normalization เท่าไร
- threshold ที่แนะนำเท่าไร
- checkpoint มาจาก dataset ใด

มี dataset fingerprint สำหรับ audit และป้องกันการใช้ config ผิดชุด

In [ ]:
# ============================================================
# 16 — SAVE TRAINING CONFIG
# ============================================================

fingerprint_payload = json.dumps(
    {
        "sample_shape": reference_info["sample_shape"],
        "timestamps": DATASET_TIMESTAMPS,
        "bands": DATASET_BANDS,
        "split_counts": split_counts,
    },
    sort_keys=True,
).encode("utf-8")

DATASET_FINGERPRINT = hashlib.sha256(
    fingerprint_payload
).hexdigest()[:16]


training_config = {
    "project": "TerraMind Rice Detection",
    "run_name": RUN_NAME,
    "terratorch_version": package_version("terratorch"),
    "model_factory": "EncoderDecoderFactory",
    "model_args": MODEL_ARGS,
    "timestamps": DATASET_TIMESTAMPS,
    "bands": DATASET_BANDS,
    "normalization": {
        "means": BAND_MEANS.tolist(),
        "stds": BAND_STDS.tolist(),
    },
    "label_values": {
        "0": "non_rice",
        "1": "rice",
        "255": "ignore",
    },
    "ignore_index": 255,
    "temporal_selection": TEMPORAL_SELECTION,
    "top_quality_timestamps": TOP_QUALITY_TIMESTAMPS,
    "loss": {
        "ce": 0.60,
        "dice": 0.40,
    },
    "class_weights": CLASS_WEIGHTS,
    "base_learning_rate": BASE_LEARNING_RATE,
    "backbone_learning_rate": BACKBONE_LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "training_mode": TRAINING_MODE,
    "resume_checkpoint": (
        str(RESUME_CHECKPOINT_PATH)
        if RESUME_CHECKPOINT_PATH
        else None
    ),
    "resume_epoch": RESUME_EPOCH,
    "max_epochs_total": MAX_EPOCHS_TOTAL,
    "additional_epochs": ADDITIONAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "accumulate_grad_batches": ACCUMULATE_GRAD_BATCHES,
    "precision": TRAIN_PRECISION,
    "recommended_probability_threshold": 0.50,
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "split_counts": split_counts,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}


LOCAL_TRAINING_CONFIG_PATH = (
    LOCAL_CONFIG_DIR / "training_config.json"
)

LOCAL_NORMALIZATION_PATH = (
    LOCAL_CONFIG_DIR / "normalization_stats.json"
)

with open(
    LOCAL_TRAINING_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

with open(
    LOCAL_NORMALIZATION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        normalization_stats,
        file,
        ensure_ascii=False,
        indent=2,
    )


shutil.copy2(
    LOCAL_TRAINING_CONFIG_PATH,
    OUTPUT_DIR / "training_config.json",
)

shutil.copy2(
    LOCAL_NORMALIZATION_PATH,
    OUTPUT_DIR / "normalization_stats.json",
)

print("Dataset fingerprint:", DATASET_FINGERPRINT)
print("Configuration saved")

# 16 — Trainer

Trainer ใช้:

- mixed precision ตาม GPU
- gradient accumulation เพื่อรักษา effective batch
- gradient clipping
- validation ทุก epoch
- CSV และ TensorBoard logs
- checkpoint best + last
- early stopping

เมื่อ resume, `ckpt_path` จะ restore weights, optimizer, scheduler และ global step

In [ ]:
# ============================================================
# 17 — TRAINER
# ============================================================

csv_logger = CSVLogger(
    save_dir=LOCAL_LOG_DIR,
    name="csv",
)

tensorboard_logger = TensorBoardLogger(
    save_dir=LOCAL_LOG_DIR,
    name="tensorboard",
)

trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    strategy="auto",
    max_epochs=MAX_EPOCHS_TOTAL,
    precision=TRAIN_PRECISION,
    accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm",
    callbacks=[
        best_rice_callback,
        early_stopping,
        lr_monitor,
        drive_backup_callback,
    ],
    logger=[
        csv_logger,
        tensorboard_logger,
    ],
    log_every_n_steps=10,
    check_val_every_n_epoch=1,
    num_sanity_val_steps=2,
    enable_progress_bar=True,
    benchmark=True,
    deterministic=False,
)

print("Trainer ready")

# 17 — เริ่ม Fine-tuning

Showcase default จะ resume ต่อ 3 epochs

ระหว่าง train:

- อย่าปิด browser tab โดยไม่จำเป็น
- checkpoint ถูกสำรองไป Drive ทุก validation epoch
- หาก runtime หลุด ให้รันใหม่และใช้ `TRAINING_MODE="resume"`
- `last.ckpt` จะพากลับไปยัง epoch ล่าสุด

In [ ]:
# ============================================================
# 18 — FIT
# ============================================================

model.train()

trainer.fit(
    model=model,
    datamodule=data_module,
    ckpt_path=(
        str(RESUME_CHECKPOINT_PATH)
        if USE_RESUME
        else None
    ),
)

drive_backup_callback.backup()

print("Training finished")
print("Best local checkpoint:", best_rice_callback.best_model_path)
print("Canonical Drive best:", OUTPUT_DIR / "best.ckpt")
print("Canonical Drive last:", OUTPUT_DIR / "last.ckpt")

# 18 — เลือก checkpoint ที่ดีที่สุดและ test

หลัง train:

1. ใช้ `best.ckpt` เป็นอันดับแรก
2. fallback เป็น local best
3. fallback เป็น `last.ckpt`

Test split ต้องไม่ถูกใช้เลือก hyperparameters  
Metric ของโครงการนี้วัดเทียบกับ labels ใน HDF5 ซึ่งปัจจุบันอาจเป็น pseudo-label จึงควรแยกจาก independent ground-truth assessment

In [ ]:
# ============================================================
# 19 — TEST BEST CHECKPOINT
# ============================================================

checkpoint_candidates = [
    OUTPUT_DIR / "best.ckpt",
    Path(best_rice_callback.best_model_path)
    if best_rice_callback.best_model_path
    else None,
    OUTPUT_DIR / "last.ckpt",
]

BEST_CHECKPOINT_PATH = next(
    (
        path
        for path in checkpoint_candidates
        if path is not None and path.exists()
    ),
    None,
)

if BEST_CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "ไม่พบ checkpoint หลัง training"
    )

print("Selected best checkpoint:", BEST_CHECKPOINT_PATH)

test_results = trainer.test(
    model=model,
    datamodule=data_module,
    ckpt_path=str(BEST_CHECKPOINT_PATH),
)

TEST_METRICS_PATH = (
    LOCAL_CONFIG_DIR / "test_metrics.json"
)

with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        test_results,
        file,
        ensure_ascii=False,
        indent=2,
    )

shutil.copy2(
    TEST_METRICS_PATH,
    OUTPUT_DIR / TEST_METRICS_PATH.name,
)

display(pd.DataFrame(test_results))

# 19 — Threshold calibration

Semantic segmentation checkpoint ให้ logits สองคลาส แต่ output mask ต้องเลือก probability threshold

Cell นี้ทดสอบ thresholds หลายค่าโดยใช้ validation set และคำนวณ:

- IoU
- Dice
- Precision
- Recall

Threshold ที่ IoU สูงสุดจะถูกบันทึกลง `training_config.json` และ Prediction Notebook จะใช้โดยอัตโนมัติ

> Calibration นี้เหมาะกับ distribution ของ validation set หากนำ model ไปพื้นที่หรือฤดูกาลใหม่ควรตรวจ threshold อีกครั้ง

In [ ]:
# ============================================================
# 20 — THRESHOLD CALIBRATION
# ============================================================

def load_model_for_inference(
    checkpoint_path: Path,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    hparams = checkpoint.get(
        "hyper_parameters",
        {},
    )

    model_args = copy.deepcopy(
        hparams.get(
            "model_args",
            MODEL_ARGS,
        )
    )

    if "backbone_pretrained" in model_args:
        model_args["backbone_pretrained"] = False

    load_kwargs = {
        "model_args": model_args,
    }

    if "model_factory" in hparams:
        load_kwargs["model_factory"] = hparams["model_factory"]

    del checkpoint
    gc.collect()

    loaded_model = (
        terratorch.tasks.SemanticSegmentationTask
        .load_from_checkpoint(
            str(checkpoint_path),
            map_location="cpu",
            strict=True,
            **load_kwargs,
        )
    )

    return loaded_model


if CALIBRATE_THRESHOLD:
    calibration_model = load_model_for_inference(
        BEST_CHECKPOINT_PATH
    ).to(DEVICE)

    calibration_model.eval()
    data_module.setup("fit")

    thresholds = np.asarray(
        THRESHOLD_CANDIDATES,
        dtype=np.float32,
    )

    tp = np.zeros(len(thresholds), dtype=np.int64)
    fp = np.zeros(len(thresholds), dtype=np.int64)
    fn = np.zeros(len(thresholds), dtype=np.int64)

    with torch.inference_mode():
        for batch in tqdm(
            data_module.val_dataloader(),
            desc="Calibrating threshold",
        ):
            images = batch["image"].to(
                DEVICE,
                non_blocking=True,
            )

            labels = batch["mask"].numpy()

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
            ):
                result = calibration_model(images)
                logits = (
                    result.output
                    if hasattr(result, "output")
                    else result
                )

                probabilities = torch.softmax(
                    logits,
                    dim=1,
                )[:, 1].float().cpu().numpy()

            valid = labels != 255
            truth = labels == 1

            for index, threshold in enumerate(thresholds):
                prediction = probabilities >= threshold

                tp[index] += int(
                    np.count_nonzero(
                        prediction & truth & valid
                    )
                )

                fp[index] += int(
                    np.count_nonzero(
                        prediction & ~truth & valid
                    )
                )

                fn[index] += int(
                    np.count_nonzero(
                        ~prediction & truth & valid
                    )
                )

    epsilon = 1e-9

    iou = tp / (tp + fp + fn + epsilon)
    dice = 2 * tp / (2 * tp + fp + fn + epsilon)
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)

    threshold_table = pd.DataFrame({
        "threshold": thresholds,
        "IoU_rice": iou,
        "Dice_rice": dice,
        "Precision_rice": precision,
        "Recall_rice": recall,
    })

    best_threshold_index = int(
        np.argmax(iou)
    )

    RECOMMENDED_THRESHOLD = float(
        thresholds[best_threshold_index]
    )

    display(threshold_table)

    THRESHOLD_METRICS_PATH = (
        OUTPUT_DIR / "threshold_calibration.csv"
    )

    threshold_table.to_csv(
        THRESHOLD_METRICS_PATH,
        index=False,
    )

    calibration_model = calibration_model.cpu()

    del calibration_model
    gc.collect()
    torch.cuda.empty_cache()

else:
    RECOMMENDED_THRESHOLD = 0.50


with open(
    OUTPUT_DIR / "training_config.json",
    "r",
    encoding="utf-8",
) as file:
    final_training_config = json.load(file)

final_training_config[
    "recommended_probability_threshold"
] = RECOMMENDED_THRESHOLD

with open(
    OUTPUT_DIR / "training_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_training_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Recommended threshold:", RECOMMENDED_THRESHOLD)

# 20 — Optional visual preview

Preview แสดง:

1. RGB
2. Label
3. Rice probability
4. Prediction ตาม calibrated threshold

ปิดได้ด้วย:

```python
SHOW_SAMPLE_PREVIEW = False
```

In [ ]:
# ============================================================
# 21 — OPTIONAL SAMPLE PREVIEW
# ============================================================

def stretch_rgb(rgb: np.ndarray) -> np.ndarray:
    output = np.zeros_like(rgb, dtype=np.float32)

    for channel in range(3):
        low, high = np.nanpercentile(
            rgb[..., channel],
            [2, 98],
        )

        output[..., channel] = np.clip(
            (rgb[..., channel] - low)
            / max(high - low, 1e-6),
            0,
            1,
        )

    return output


if SHOW_SAMPLE_PREVIEW:
    preview_model = load_model_for_inference(
        BEST_CHECKPOINT_PATH
    ).to(DEVICE)

    preview_model.eval()
    data_module.setup("test")

    batch = next(
        iter(data_module.test_dataloader())
    )

    images = batch["image"].to(DEVICE)
    labels = batch["mask"].numpy()

    with torch.inference_mode():
        result = preview_model(images)
        logits = (
            result.output
            if hasattr(result, "output")
            else result
        )

        probabilities = torch.softmax(
            logits,
            dim=1,
        )[:, 1].float().cpu().numpy()

    predictions = (
        probabilities >= RECOMMENDED_THRESHOLD
    ).astype(np.uint8)

    images_cpu = images.cpu().numpy()
    sample_count = min(4, images_cpu.shape[0])

    fig, axes = plt.subplots(
        sample_count,
        4,
        figsize=(16, 4 * sample_count),
    )

    if sample_count == 1:
        axes = np.asarray([axes])

    for index in range(sample_count):
        middle_time = images_cpu.shape[2] // 2

        rgb = images_cpu[
            index,
            [2, 1, 0],
            middle_time,
        ]

        rgb = (
            rgb
            * BAND_STDS[[2, 1, 0], None, None]
            + BAND_MEANS[[2, 1, 0], None, None]
        )

        rgb = np.transpose(rgb, (1, 2, 0))

        axes[index, 0].imshow(stretch_rgb(rgb))
        axes[index, 0].set_title("RGB")
        axes[index, 0].axis("off")

        label_show = labels[index].astype(np.float32)
        label_show[labels[index] == 255] = np.nan

        axes[index, 1].imshow(
            label_show,
            vmin=0,
            vmax=1,
        )
        axes[index, 1].set_title("Label")
        axes[index, 1].axis("off")

        axes[index, 2].imshow(
            probabilities[index],
            vmin=0,
            vmax=1,
        )
        axes[index, 2].set_title("Rice probability")
        axes[index, 2].axis("off")

        axes[index, 3].imshow(
            predictions[index],
            vmin=0,
            vmax=1,
        )
        axes[index, 3].set_title(
            f"Prediction ≥ {RECOMMENDED_THRESHOLD:.2f}"
        )
        axes[index, 3].axis("off")

    plt.tight_layout()
    plt.show()

    preview_model = preview_model.cpu()

    del (
        preview_model,
        images,
        logits,
        result,
    )

    gc.collect()
    torch.cuda.empty_cache()

else:
    print("Preview disabled")

# 21 — Export logs และสรุป

CSV Logger ของ Lightning เก็บ metrics ราย step/epoch  
Cell นี้คัดลอก training history และ environment manifest ไป Google Drive

In [ ]:
# ============================================================
# 22 — EXPORT LOGS + FINAL SUMMARY
# ============================================================

metrics_candidates = list(
    LOCAL_LOG_DIR.rglob("metrics.csv")
)

if metrics_candidates:
    training_history_source = metrics_candidates[-1]

    shutil.copy2(
        training_history_source,
        OUTPUT_DIR / "training_history.csv",
    )


environment_manifest = {
    "numpy": np.__version__,
    "pytorch": torch.__version__,
    "lightning": pl.__version__,
    "terratorch": package_version("terratorch"),
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "precision": TRAIN_PRECISION,
    "project_root": str(PROJECT_ROOT),
    "dataset_dir": str(DATASET_DIR),
    "model_dir": str(MODEL_DIR),
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(
    OUTPUT_DIR / "environment.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )


output_files = [
    "best.ckpt",
    "last.ckpt",
    "training_config.json",
    "normalization_stats.json",
    "test_metrics.json",
    "training_history.csv",
    "environment.json",
]

print("=" * 80)
print("FINE-TUNING WORKFLOW COMPLETE")
print("=" * 80)
print("Best checkpoint:", OUTPUT_DIR / "best.ckpt")
print("Last checkpoint:", OUTPUT_DIR / "last.ckpt")
print("Recommended threshold:", RECOMMENDED_THRESHOLD)

print("\nOutput status:")

for filename in output_files:
    path = OUTPUT_DIR / filename

    print(
        filename,
        "OK" if path.exists() else "NOT CREATED",
        (
            f"| {path.stat().st_size / 1024**2:.2f} MB"
            if path.exists()
            else ""
        ),
    )

# Troubleshooting guide

## `ResolutionImpossible` ระหว่างติดตั้ง

อย่า pin NumPy เป็น `1.26.4` คู่กับ TerraTorch 1.2.10  
Notebook นี้ไม่ pin NumPy

## `terratorch has no attribute __version__`

ใช้:

```python
from importlib.metadata import version
version("terratorch")
```

## PyTorch แสดง `+cpu`

เลือก GPU Runtime แล้ว restart session  
Notebook จะหยุดตั้งแต่ต้นหาก `torch.cuda.is_available()` เป็น False

## Hugging Face HTTP 504

ระบบ retry และ cache ให้แล้ว  
Public model ไม่จำเป็นต้องมี token แต่ `HF_TOKEN` ใน Colab Secrets ช่วยเพิ่ม rate limit

## Resume แล้ว dataset ไม่ตรง checkpoint

ห้ามฝืน resume ให้เปลี่ยน:

```python
TRAINING_MODE = "pretrained"
```

## Runtime หลุด

รันใหม่จากบนลงล่างและใช้:

```python
TRAINING_MODE = "resume"
```

ระบบจะเลือก `last.ckpt`

## Modules อยู่ใน eval mode

Smoke test จบด้วย `model.train()` แล้ว

## Metric สูงแต่ผลจริงยังไม่เท่ากัน

Validation/test อาจวัดกับ pseudo-label  
ควรมี independent ground truth หรือ visual QA เพิ่มเติม

# References

- TerraTorch official repository: https://github.com/torchgeo/terratorch
- TerraMind model card: https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-base
- TerraMind paper: https://arxiv.org/abs/2504.11171
- TerraTorch paper: https://arxiv.org/abs/2503.20563
- Lightning checkpointing: https://lightning.ai/docs/pytorch/stable/common/checkpointing_basic.html

TerraTorch และ TerraMind ใช้ license ตาม repository/model card ของแต่ละโครงการ ผู้ใช้งานต้องตรวจ license ให้ตรงกับการนำไปใช้จริงของบริษัท